<a href="https://colab.research.google.com/github/aniket-alt/unsloth/blob/main/1_Full_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U unsloth transformers trl datasets accelerate peft bitsandbytes xformers gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 21.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavio

In [2]:
# ==== System setup (Colab/Kaggle) ====

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
BF16 = is_bfloat16_supported() # autocasts to bf16 if available

# Utility: tiny evaluation helper
def chat(model, tokenizer, user, system="You are a helpful assistant.", max_new_tokens=128):
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    # 1. Tokenize. This returns a single PyTorch Tensor.
    inputs_tensor = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    )

    # 2. Move the tensor to the model's device
    inputs_on_device = inputs_tensor.to(model.device)

    from transformers import TextStreamer
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        # 3. Call generate.
        # We MUST pass `input_ids` as the first argument to avoid
        # a 'KeyError: key' bug in this version of Unsloth.
        _ = model.generate(
            input_ids = inputs_on_device, # <--- This must be the first argument
            streamer = streamer,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = 0.7
        )

print("Setup complete.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Setup complete.


In [3]:
# ==== A. Full finetuning (SmolLM2‑135M) ====
MODEL = "unsloth/SmolLM2-135M-Instruct"  # tiny & fast
MAX_SEQ = 1024

# 1) Load model for full finetune
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=False,
    full_finetuning=True,   # <— Full‑parameter finetune
    dtype=torch.bfloat16 if BF16 else torch.float16,
)

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

In [4]:
# 2) Dataset: small, clean instruction tuning data
ds = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")  # small slice

def to_text(batch):
    texts = []
    for ins, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user = ins if not inp else f"{ins}\n\n{inp}"
        msgs = [
            {"role": "system","content":"You are a helpful assistant."},
            {"role":"user","content":user},
            {"role":"assistant","content":out},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False))
    return {"text": texts}

train = ds.map(to_text, batched=True, remove_columns=ds.column_names)

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
# 3) Train with TRL SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    bf16=BF16,
    fp16=not BF16,
    logging_steps=10,
    save_steps=100,
    output_dir="fft-smollm2",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=True,
    args=args,
)
trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 134,515,584 of 134,515,584 (100.00% trained)


Step,Training Loss
10,1.518500
20,1.419800
30,1.433500
40,1.378400
50,1.466900
60,1.383100
70,1.379900
80,1.420300
90,1.419900
100,1.383000


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=125, training_loss=1.4276554832458497, metrics={'train_runtime': 246.5842, 'train_samples_per_second': 8.111, 'train_steps_per_second': 0.507, 'total_flos': 414068583135744.0, 'train_loss': 1.4276554832458497, 'epoch': 1.0})

In [6]:
# 4) Test
from unsloth import FastLanguageModel
import torch

target_dtype = torch.bfloat16 if BF16 else torch.float16
print(f"Manually casting model to {target_dtype} for inference...")

model.to(target_dtype)

FastLanguageModel.for_inference(model)
print("\n=== Test ===")
chat(model, tokenizer, "Give me 3 creative ice-breaker questions for a meetup.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Manually casting model to torch.float16 for inference...

=== Test ===
Here are three creative ice-breaker questions for a meetup:

1. How do you like to spend your downtime?
2. How many pets do you own?
3. What do you like most about your job?

These questions will help you have a conversation with the new co-worker or manager, helping to break the ice and get to know them better.


In [7]:
# ==== Minimal Gradio chat ====
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread
FastLanguageModel.for_inference(model)

def respond(message, history):
    msgs = []
    for u,a in history + [(message,"")]:
        msgs.append({"role":"user","content":u})
        msgs.append({"role":"assistant","content":a})
    msgs.pop()  # drop trailing empty assistant
    input_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    def run_gen(): model.generate(input_ids=input_ids, streamer=streamer, max_new_tokens=256, temperature=0.7, do_sample=True)
    Thread(target=run_gen).start()
    partial=""
    for token in streamer:
        partial += token
        yield partial

gr.ChatInterface(respond, title="Unsloth Chat UI").launch(share=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>